# Compression, Materialized Views, and Inverted Indexes

**Module 6: Table Properties, Materialized Views, and Indexes** · [Quiz](quiz06_properties_mv_and_indexes.ipynb) · [Course README](../../README.md)

This Lab works on the telemetry that the earlier modules loaded into Doris: the logs in `otel_logs` and the gauge metrics in `otel_metrics_gauge`.

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


## Check the logs in Doris

The examples below read `otel_logs`: the demo's logs from Module 1 and the FE log from Module 2. Count the rows by service:

In [ ]:
log_services = lab.wait_for_rows(
    """
    SELECT service_name, COUNT(*) AS log_rows, MAX(timestamp) AS latest_log
    FROM otel_logs
    GROUP BY service_name
    ORDER BY log_rows DESC
    """,
    title="Log rows by service in otel_logs",
    hint="otel_logs has no rows. Start the DOG stack and the demo with the cells in Module 1, "
         "and load the FE log with section 1 of Module 2. Then rerun this cell.",
)

**Expected result**

One row per service: `doris-fe`, the FE log that Module 2 loaded, and the demo's services, such as `frontend-proxy`, `kafka`, `currency`, and `cart`. The inverted-index example at the end counts the rows of `cart`, which logs as long as the demo runs. If `doris-fe` is the only service, the demo has not sent logs: start it with `lab.start_demo()` in Module 1 and rerun this cell after a minute.

## Compress

Observability systems may receive a very large amount of data every day. In addition to data governance, what other methods can help reduce disk usage? One recommended approach is to use the ZSTD compression algorithm to compress data efficiently and reduce storage consumption.

The logs have already been imported into the `otel_logs` table. That table uses the default compression algorithm, LZ4F. To compare it with the default `otel_logs` table, create a new table named `otel_logs_zstd`:

In [ ]:
lab.execute(r"""
CREATE TABLE `otel_logs_zstd` (
  `timestamp` datetime(6) NULL,
  `service_name` varchar(200) NULL,
  `service_instance_id` varchar(200) NULL,
  `trace_id` varchar(200) NULL,
  `span_id` text NULL,
  `severity_number` int NULL,
  `severity_text` text NULL,
  `body` text NULL,
  `resource_attributes` variant NULL,
  `log_attributes` variant NULL,
  `scope_name` text NULL,
  `scope_version` text NULL,
  INDEX idx_service_name (`service_name`) USING INVERTED,
  INDEX idx_timestamp (`timestamp`) USING INVERTED,
  INDEX idx_service_instance_id (`service_instance_id`) USING INVERTED,
  INDEX idx_trace_id (`trace_id`) USING INVERTED,
  INDEX idx_span_id (`span_id`) USING INVERTED,
  INDEX idx_severity_number (`severity_number`) USING INVERTED,
  INDEX idx_body (`body`) USING INVERTED PROPERTIES("lower_case" = "true", "parser" = "unicode", "support_phrase" = "true"),
  INDEX idx_severity_text (`severity_text`) USING INVERTED,
  INDEX idx_resource_attributes (`resource_attributes`) USING INVERTED,
  INDEX idx_log_attributes (`log_attributes`) USING INVERTED,
  INDEX idx_scope_name (`scope_name`) USING INVERTED,
  INDEX idx_scope_version (`scope_version`) USING INVERTED
) ENGINE=OLAP
DUPLICATE KEY(`timestamp`, `service_name`)
PARTITION BY RANGE(`timestamp`)
(PARTITION p20260908 VALUES [('2026-09-08 00:00:00'), ('2026-09-09 00:00:00')),
PARTITION p20260909 VALUES [('2026-09-09 00:00:00'), ('2026-09-10 00:00:00')),
PARTITION p20260910 VALUES [('2026-09-10 00:00:00'), ('2026-09-11 00:00:00')),
PARTITION p20260911 VALUES [('2026-09-11 00:00:00'), ('2026-09-12 00:00:00')),
PARTITION p20260912 VALUES [('2026-09-12 00:00:00'), ('2026-09-13 00:00:00')),
PARTITION p20260913 VALUES [('2026-09-13 00:00:00'), ('2026-09-14 00:00:00')),
PARTITION p20260914 VALUES [('2026-09-14 00:00:00'), ('2026-09-15 00:00:00')),
PARTITION p20260915 VALUES [('2026-09-15 00:00:00'), ('2026-09-16 00:00:00')),
PARTITION p20260916 VALUES [('2026-09-16 00:00:00'), ('2026-09-17 00:00:00')),
PARTITION p20260917 VALUES [('2026-09-17 00:00:00'), ('2026-09-18 00:00:00')),
PARTITION p20260918 VALUES [('2026-09-18 00:00:00'), ('2026-09-19 00:00:00')),
PARTITION p20260919 VALUES [('2026-09-19 00:00:00'), ('2026-09-20 00:00:00')))
DISTRIBUTED BY RANDOM BUCKETS AUTO
PROPERTIES (
"replication_allocation" = "tag.location.default: 1",
"min_load_replica_num" = "-1",
"is_being_synced" = "false",
"dynamic_partition.enable" = "true",
"dynamic_partition.time_unit" = "DAY",
"dynamic_partition.time_zone" = "Etc/UTC",
"dynamic_partition.start" = "-10",
"dynamic_partition.end" = "1",
"dynamic_partition.prefix" = "p",
"dynamic_partition.replication_allocation" = "tag.location.default: 1",
"dynamic_partition.buckets" = "10",
"dynamic_partition.create_history_partition" = "true",
"dynamic_partition.history_partition_num" = "10",
"dynamic_partition.hot_partition_num" = "0",
"dynamic_partition.reserved_history_periods" = "NULL",
"dynamic_partition.storage_policy" = "",
"storage_medium" = "hdd",
"storage_format" = "V2",
"inverted_index_storage_format" = "V2",
"light_schema_change" = "true",
"compaction_policy" = "time_series",
"time_series_compaction_goal_size_mbytes" = "1024",
"time_series_compaction_file_count_threshold" = "2000",
"time_series_compaction_time_threshold_seconds" = "3600",
"time_series_compaction_empty_rowsets_threshold" = "5",
"time_series_compaction_level_threshold" = "1",
"disable_auto_compaction" = "false",
"group_commit_interval_ms" = "10000",
"group_commit_data_bytes" = "134217728",
"compression" = "zstd"
);
""", title='ZSTD-compressed log table')


Note the final property, `"compression" = "zstd"`. It specifies that the ZSTD compression algorithm is used.

Next, insert the data from `otel_logs` into `otel_logs_zstd`:

In [ ]:
lab.execute("INSERT INTO otel_logs_zstd SELECT * FROM otel_logs")


Because the data files are not large, this example is intended only to show the difference. In scenarios with larger data volumes, ZSTD can provide better compression results:

<img src="images/compress.png" alt="Comparison of LZ4F and ZSTD compression in Doris" style="max-width:100%;height:auto;">

## Materialized Views

In metrics scenarios, if you need to analyze the maximum value of certain `metric_name` values over a historical period, you can query the `otel_metrics_gauge` table directly. However, this table stores the original metrics data, and direct aggregation can be inefficient. Doris materialized views can be used to accelerate these queries.

The `otel_metrics_gauge` table has already been created. You only need to create a materialized view on this table to accelerate the required query:

In [ ]:
lab.execute(r"""
CREATE MATERIALIZED VIEW mv_for_metrics AS
SELECT metric_name AS mn, MAX(value) AS max_value
FROM otel_metrics_gauge
GROUP BY metric_name;
""", title='Materialized view for metric maxima')


After the materialized view is created, you can query the original table without needing to remember that the materialized view exists:

In [ ]:
lab.sql(r"""
SELECT metric_name, MAX(value) FROM otel_metrics_gauge GROUP BY metric_name LIMIT 5;
""", title='Metric maximum query')


This statement will hit the materialized view. Because the materialized view already contains aggregated data, the query will be accelerated. You can use `EXPLAIN` to confirm this:

<img src="images/explain.png" alt="EXPLAIN output showing that the materialized view is used" style="max-width:100%;height:auto;">

## Inverted Indexes

The exporter created `otel_logs` with an inverted index on every column. An inverted index lists, for each value of a column, the rows that contain it; for the log message in `body`, it lists the rows that contain each word. With it, Doris can find the rows that match a filter such as `service_name = 'cart'` without reading the column. List the indexes of `otel_logs`:

In [ ]:
lab.sql("SHOW INDEX FROM otel_logs", title="Inverted indexes of otel_logs",
        columns=["Key_name", "Column_name", "Index_type", "Properties"]);

**Expected result**

12 indexes, all `INVERTED`, one per column, such as `idx_service_name` on `service_name`. Only `idx_body` has properties: `"parser" = "unicode"` splits each message into words, `"lower_case" = "true"` indexes the words in lower case, and `"support_phrase" = "true"` also records where each word occurs. With them, queries can search the messages for words with `MATCH_ANY` and for phrases with `MATCH_PHRASE`. The other indexes keep each value whole, for filters such as `=` and `IN`, and for ranges of numbers and times.

### Measure what the index filters

Does Doris use the index of `service_name`, and how much reading does it save? A Query Profile shows it. For each operator of a query, such as the scan of a table, Doris records what the operator did, including how many rows the scan skipped and why. Doris collects a profile for every query of a session in which the variable `enable_profile` is `true`. The next cell sets it for this notebook's session, together with three related variables:

- `profile_level = 2`: the profile includes the row counters of the scan. This is the default in Doris 4.1; level 1 leaves most of them out.
- `enable_sql_cache = false`: when a table has not changed for a while, the SQL cache can return the result of a repeated query without scanning the table, and the profile of that query shows no scan.
- `enable_condition_cache = false`: the condition cache remembers which parts of the data had no matching rows in an earlier run of a filter, and a rerun skips them, so the counters of the rerun no longer show all the rows that the index filters.

In [ ]:
lab.execute("""
SET enable_profile = true,
    profile_level = 2,
    enable_sql_cache = false,
    enable_condition_cache = false
""", title="Collect Query Profiles in this session, and scan the table on every run");

**Expected result**

A card says that the statement succeeded. The variables apply to this notebook's connection to Doris until the kernel restarts; after a restart, run the first cell of this Lab and this cell again. In a MySQL client, run the same `SET` statement before your queries.

Count the log rows of `cart`, one of the demo's services, by severity, and fetch the profile of the query. `lab.profile_query(...)` runs the query, then finds and fetches its profile in three steps, which you can also take in a MySQL client and a terminal:

1. `SELECT last_query_id()` returns the ID of the last query in the session. The ID of a query is also the ID of its profile.
2. `SHOW QUERY PROFILE` lists the newest profiles, each with its ID and SQL. When the query's ID is in the list, Doris has collected its profile.
3. The FE HTTP API returns the text of a profile by its ID. The cell shows the request as a `curl` command.

The FE web UI lists the same profiles: open http://127.0.0.1:28030 (port `DORIS_OBS_HTTP_PORT`), log in as `root` with no password, and open QueryProfile.

In [ ]:
INDEX_QUERY = """
SELECT severity_text, COUNT(*) AS log_rows
FROM otel_logs
WHERE service_name = 'cart'
GROUP BY severity_text
ORDER BY log_rows DESC
"""
profile = lab.profile_query(INDEX_QUERY, title="Log rows of cart by severity")

**Expected result**

The result first: one or two rows. Almost all log rows of `cart` have the severity `Information`, the label that .NET services write, and one may have `Warning`. `cart` logs as long as the demo runs, so the counts grow whenever you rerun the cell.

Then the three steps: the query ID, the query's row in `SHOW QUERY PROFILE` with its start time and duration (`Total`, a few milliseconds), and the `curl` command, followed by the profile text in a collapsed box, a few thousand lines long. The text starts with a `Summary` of the query, and `ChangedSessionVariables` lists the variables that the `SET` cell changed from their defaults. Then the operators of the query plan follow twice. Doris runs each operator as several instances in parallel: `MergedProfile` combines the instances of each operator, and `DetailProfile` lists every instance on its own. The row counters of the table scan are in `DetailProfile`, in each `OLAP_SCAN_OPERATOR` under `SegmentIterator`: search the text for `RowsInvertedIndexFiltered`.

Each row of the table that the scan considers is counted once, by the step that removed it, or as a row that passed: skipped by the zone maps, skipped by the inverted index, read and dropped by the filter, or read and kept. Add up these counters over the scan instances:

In [ ]:
with_index = lab.scan_filters(profile, title="How the scan of otel_logs filtered its rows")
assert with_index["RowsInvertedIndexFiltered"] > 0, (
    "The inverted index filtered no rows. Check that otel_logs has logs of several demo services "
    "(the first cell of this Lab), then rerun the query and the cells after it."
)

**Expected result**

Four counters, which add up to the number of rows that `otel_logs` had when the query ran:

- `RowsStatsFiltered`: the rows of `doris-fe`, or most of them. Doris stores the rows of a table in segments (data files), sorted by the key, and keeps the minimum and maximum of each column for every segment and for every page of rows in it: the zone maps. Module 2 loaded the earlier lines of the FE log in large batches, so they fill segments and pages of their own, in which `service_name` is `doris-fe` from the minimum to the maximum. These cannot hold `cart`, so Doris skips them without reading them.
- `RowsInvertedIndexFiltered`: most of the other rows, about two thousand when you run the Labs one after another, and tens of thousands once the demo has run for half an hour. The demo's services log at the same time, so their rows share segments and pages, and the range of `service_name` in them includes `cart`. In these, the index lists the rows of `cart`, and Doris leaves out all other rows before it reads any data.
- `RowsVectorPredFiltered`: 0. These would be rows that Doris reads and then drops, after it compares their value with the filter; with the index, Doris reads only matching rows.
- `RowsRead`: the rows of `cart`, as many as the `log_rows` of the query add up to. Doris read their `severity_text` to group them.

Now measure the same query without the index. With `enable_inverted_index_query = false`, Doris evaluates filters without inverted indexes. The next cell turns the index off for this session, runs the query and adds up the counters of its profile, and turns the index on again:

In [ ]:
import pandas as pd

lab.execute("SET enable_inverted_index_query = false", title="Stop using inverted indexes in this session")
try:
    profile_without_index = lab.profile_query(INDEX_QUERY, title="Log rows of cart by severity, without the index")
    without_index = lab.scan_filters(profile_without_index, title="How the scan filtered the rows without the index")
finally:
    # Use the index again, also when a step above failed.
    lab.execute("SET enable_inverted_index_query = true", title="Use inverted indexes again")

pd.DataFrame({"with the index": with_index, "without the index": without_index})

**Expected result**

The query returns the same counts as before, plus the rows that arrived in between. The last table puts the two measurements side by side:

- `RowsStatsFiltered` is about the same: the zone maps do not depend on the inverted index.
- `RowsInvertedIndexFiltered` is 0, and about as many rows appear under `RowsVectorPredFiltered` instead: Doris read the `service_name` of each of these rows and compared it with `'cart'`.
- `RowsRead` is about the same: the same rows match.

Without the index, Doris read `RowsVectorPredFiltered` plus `RowsRead` rows to find the `RowsRead` rows of `cart`: many times as many rows as with the index. In a table this small, both queries take milliseconds; in a table of billions of log rows, reading and comparing the extra rows costs disk reads and CPU time on every query.

`RowsInvertedIndexFiltered` is evidence of what the index did in this scan of this query: it removed these rows before Doris read them. On its own, it does not show how much faster the query became, what else the query spent its time on, or whether another filter can use an index. Measure each query that you want to make such a claim about.

## Next

Open [Quiz 6: Table Properties, Materialized Views, and Indexes](quiz06_properties_mv_and_indexes.ipynb), then continue to [Module 7: Trace Analysis and Litefuse](../module07-data-analysis-litefuse/lab07_data_analysis_litefuse.ipynb).